# **Day 72: asyncio Basics (Doing Many Things Together, Part 3)**

**asyncio** is another way to do many **waiting** jobs together. It uses only **one thread**, but it **switches** to another job whenever the current job is waiting.

**Real-life example:** A **waiter in a restaurant**. He takes an order from table 1, and while the kitchen is cooking, he goes to table 2, then table 3. He does not stand still in the kitchen waiting for table 1's food.

**New keywords:**
- `async def`: makes a special function called a **coroutine**.
- `await`: means "wait here for this to finish, and meanwhile let other jobs run".

In [1]:
import asyncio, time

async def greet(name):
    print(f"Hello {name}")
    await asyncio.sleep(1)          # Wait without blocking other jobs
    print(f"Bye {name}")
    return name.upper()

coroutine = greet("Rimi")
print(coroutine)                    # Calling it does not run it, it only creates a coroutine object

<coroutine object greet at 0x000001E3E9A19E50>


## **Running Coroutines**
- In a normal `.py` script, we use `asyncio.run(main())`.
- In **Jupyter**, an event loop is already running, so we can simply write `await` directly in a cell.

In [2]:
result = await coroutine
print(result)

Hello Rimi


Bye Rimi
RIMI


## **One by One vs Together (gather)**

In [3]:
async def fetch(city, seconds):
    print(f"  -> Asking weather for {city}")
    await asyncio.sleep(seconds)
    print(f"  <- Got weather for {city}")
    return city, seconds

start = time.perf_counter()
await fetch("Kolkata", 1)
await fetch("Siliguri", 1)
await fetch("Digha", 1)
print(f"One by one: {time.perf_counter() - start:.2f} seconds")

  -> Asking weather for Kolkata


  <- Got weather for Kolkata
  -> Asking weather for Siliguri


  <- Got weather for Siliguri
  -> Asking weather for Digha


  <- Got weather for Digha
One by one: 3.04 seconds


In [4]:
start = time.perf_counter()
results = await asyncio.gather(
    fetch("Kolkata", 1),
    fetch("Siliguri", 1.5),
    fetch("Digha", 0.5),
)
print(results)
print(f"Together with gather: {time.perf_counter() - start:.2f} seconds")

  -> Asking weather for Kolkata
  -> Asking weather for Siliguri
  -> Asking weather for Digha


  <- Got weather for Digha


  <- Got weather for Kolkata


  <- Got weather for Siliguri
[('Kolkata', 1), ('Siliguri', 1.5), ('Digha', 0.5)]
Together with gather: 1.51 seconds


Let us understand the code above:
- `asyncio.gather()` starts all three jobs **together**.
- The total time is about the time of the **slowest** job (1.5 seconds), not the sum (3 seconds).
- The results come back in the **same order** we gave them.

## **Never Use time.sleep() Inside async Code**
`time.sleep()` **freezes everything**. Inside async code, always use `await asyncio.sleep()`.

In [5]:
async def bad(name):
    time.sleep(0.5)       # Blocks the whole program
    return name

start = time.perf_counter()
await asyncio.gather(bad("a"), bad("b"), bad("c"))
print(f"With time.sleep: {time.perf_counter() - start:.2f} seconds (no time saved)")

With time.sleep: 1.50 seconds (no time saved)


## **create_task: Start Now, Collect the Result Later**

In [6]:
async def boil_water():
    await asyncio.sleep(1)
    return "Water boiled"

async def cut_vegetables():
    await asyncio.sleep(0.7)
    return "Vegetables cut"

async def make_khichuri():
    start = time.perf_counter()
    water = asyncio.create_task(boil_water())         # Starts immediately
    vegetables = asyncio.create_task(cut_vegetables())
    print("Both tasks started, washing rice meanwhile...")
    await asyncio.sleep(0.3)
    print(await vegetables)
    print(await water)
    print(f"Khichuri ready in {time.perf_counter() - start:.2f} seconds")

await make_khichuri()

Both tasks started, washing rice meanwhile...


Vegetables cut


Water boiled
Khichuri ready in 1.00 seconds


## **Example from Remote Sensing: Checking Many Scene Servers**
Suppose we must ask 6 different servers whether a scene is ready to download. Each server takes some time to answer. With asyncio, we ask all of them together.

In [7]:
import random
random.seed(4)

async def check_scene(scene_id):
    delay = random.uniform(0.2, 1.0)            # Pretend network delay
    await asyncio.sleep(delay)
    ready = random.random() > 0.3
    return scene_id, ready, round(delay, 2)

scenes = [f"S2A_45QXF_202401{d:02d}" for d in range(5, 31, 5)]

start = time.perf_counter()
results = await asyncio.gather(*(check_scene(s) for s in scenes))
for sid, ready, delay in results:
    print(f"{sid}  {'READY' if ready else 'not ready':<10} (answered in {delay}s)")
print(f"Checked {len(scenes)} scenes in {time.perf_counter() - start:.2f} seconds")

S2A_45QXF_20240105  not ready  (answered in 0.39s)
S2A_45QXF_20240110  READY      (answered in 0.28s)
S2A_45QXF_20240115  READY      (answered in 0.52s)
S2A_45QXF_20240120  READY      (answered in 0.32s)
S2A_45QXF_20240125  READY      (answered in 0.25s)
S2A_45QXF_20240130  not ready  (answered in 0.52s)
Checked 6 scenes in 0.53 seconds


- `*(check_scene(s) for s in scenes)` creates one coroutine for each scene and spreads them into `gather()`.

## **How asyncio Works (Simple Picture)**
1. The **event loop** keeps a list of jobs that are ready to run.
2. It runs one job until that job says `await` on something that is not ready yet.
3. Then it **switches** to another ready job.
4. When the waited-for thing is ready (a timer ends, data arrives), that job goes back into the ready list.

In [8]:
# Quick question
# If three jobs take 2, 3 and 1 seconds, about how long will asyncio.gather() take?